In [1]:
!nvidia-smi

Tue Jun  2 11:18:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/caigaojiang/LLMOPT

fatal: destination path 'LLMOPT' already exists and is not an empty directory.


In [3]:
!pip install transformers==4.42.0 \
             accelerate==0.27.2 \
             tiktoken==0.7.0 \
             einops==0.7.0 \
             transformers_stream_generator==0.0.4 \
             peft==0.11.1 \
             trl==0.9.6 \
             Jinja2==3.1.4 \
             bitsandbytes \
             "numpy<=2.0.0 "\
             pyomo
# Install the Linux system-level math solvers that Pyomo relies on
!apt-get install -y coinor-cbc glpk-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.


In [5]:
import os
import json
import sys


import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)

import peft
import trl
from pyomo.environ import *
import pyomo.environ as aml

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Current GPU: Tesla T4


In [6]:
# Configure the 4-bit quantization layout to protect cloud VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "ant-opt/LLMOPT-Qwen2.5-14B"

# Download and cache the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Download, quantize, and load the 14B model safely
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True        # Forces shard-by-shard loading directly into RAM/VRAM
)

print("done")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

...

model-00011-of-00012.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00012-of-00012.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

done


# Task 1

In [ ]:
import sys
import io
import traceback

def execute_pyomo_code(code_string):
    """
    Executes a string of Pyomo code dynamically, ensuring the __main__ 
    guard block fires correctly.
    """
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = stdout_buffer
    sys.stderr = stderr_buffer
    
    success = False
    try:
        clean_code = code_string.replace("```python", "").replace("```", "").strip()
        
        # Explicitly pass the __main__ variable so main() actually executes
        exec_context = {"__name__": "__main__"}
        exec(clean_code, exec_context, exec_context)
        success = True
    except Exception as e:
        sys.stderr.write(traceback.format_exc())
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr
        
    return {
        "success": success,
        "stdout": stdout_buffer.getvalue().strip(),
        "stderr": stderr_buffer.getvalue().strip(),
    }

In [ ]:
dataset_path = "LLMOPT/data/testset/nl4opt_test.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems)} test items from NL4Opt.")
print(f"Sample test problem:\n{json.dumps(test_problems[0], indent=1)}")

In [ ]:
# Select the first problem from the benchmark dataset
sample_item = test_problems[0]
print(json.dumps(sample_item, indent=1))

problem_description = sample_item['question']
print(problem_description)

In [ ]:
# Build an absolute path from this notebook's parent directory
module_path = os.path.abspath('LLMOPT/prompts')
print(f"module_path: {module_path}")
# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

import generate_prompt
# reconstruct
instruction_prompt = generate_prompt.Q2F(problem_description)
print(f"instruction_prompt:\n\n{instruction_prompt}")

In [ ]:
# Tokenize and run inference through the GPU
inputs = tokenizer(instruction_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False # Forces greedy decoding 
    )

In [ ]:
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text[len(instruction_prompt):].strip())

In [ ]:
# Extract everything from the model's output starting from the first markdown header
five_element_text = generated_text[len(instruction_prompt):].strip()

# extract the five_element_text, get rid of the text that the model wrote before the five_element_text
if "## Sets:" in five_element_text:
    # Keep only from '## Sets:' onwards and clean remove ```
    five_element_text = "## Sets:" + five_element_text.split("## Sets:")[1].replace("```", '').strip()

print(f"Cleaned five_element_text:\n\n{five_element_text}")

In [ ]:
code_gen_prompt = generate_prompt.F2C(five_element_text)
print(code_gen_prompt)

In [ ]:
# Wrap the prompt inside the official Qwen chat template structure
messages = [
    {"role": "user", "content": code_gen_prompt}
]

# Format prompt
formatted_chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize 
code_inputs = tokenizer([formatted_chat_prompt], return_tensors="pt").to("cuda")

with torch.no_grad():
    code_outputs = model.generate(
        **code_inputs,
        max_new_tokens=1500,
        do_sample=False # greedy decoding
    )

In [ ]:
# Decode only the newly generated tokens
generated_tokens = code_outputs[0][len(code_inputs[0]):]
raw_code_string = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n--- GENERATED SOLVER CODE ---")
print(raw_code_string)

In [ ]:
# Strip markdown formatting ticks out of the string if they are present
clean_code_to_run = raw_code_string.replace("```python", "").replace("```", "").strip()

print("Passing generated script to the background COIN-OR CBC math solver...")
run_results = execute_pyomo_code(clean_code_to_run)
print(run_results)

if run_results["success"]:
    print("\n SOLVER EXECUTED SUCCESSFULLY!")
    print("--- STDOUT (Solution Results) ---")
    print(run_results["stdout"])
else:
    print("\n SOLVER RUNTIME/SYNTAX ERROR DETECTED!")
    print("--- STDERR (Traceback Logs) ---")
    print(run_results["stderr"])

# Self-correction Phase

since the model got this problem right in the first try, we wanna intentionally send a wrong code just to test out the self correction phase.

In [ ]:
print("Wrong code")
# Intentionally make mistake in the code 
run_results_wrong_code = execute_pyomo_code(clean_code_to_run[0:40])
print(run_results_wrong_code)

if run_results_wrong_code["success"]:
    print("\n SOLVER EXECUTED SUCCESSFULLY!")
    print("--- STDOUT (Solution Results) ---")
    print(run_results_wrong_code["stdout"])
else:
    print("\n SOLVER RUNTIME/SYNTAX ERROR DETECTED!")
    print("--- STDERR (Traceback Logs) ---")
    print(run_results_wrong_code["stderr"])

In [ ]:
import self_correction_prompt

# Test if the model is correcting the code
faulty_code = clean_code_to_run
error_log = run_results_wrong_code["stderr"]

correction_prompt_input = {
    'ques': problem_description, 
    'five': five_element_text, 
    'code': clean_code_to_run[0:40], 
    'output': run_results_wrong_code["stdout"], 
    'error': run_results_wrong_code["stderr"]
}
# Reconstruct the Self-Correction Prompt 
formatted_correction_prompt = self_correction_prompt.self_correction(**correction_prompt_input)
print(formatted_correction_prompt)

In [ ]:
correction_inputs = tokenizer([formatted_correction_prompt], return_tensors="pt").to("cuda")

with torch.no_grad():
    correction_outputs = model.generate(
        **correction_inputs,
        max_new_tokens=1536,
        do_sample=False
    )

# Decode only the newly generated tokens
corrected_tokens = correction_outputs[0][len(correction_inputs[0]):]
corrected_code_string = tokenizer.decode(corrected_tokens, skip_special_tokens=True)

print("\n--- GENERATED CORRECTED CODE ---")
print(corrected_code_string)

In [ ]:
corrected_code_string = tokenizer.decode(corrected_tokens, skip_special_tokens=True)
corrected_code_string = corrected_code_string.split("```")[1].replace("```", '').replace("python", '').strip()
print(corrected_code_string)
clean_corrected_code = corrected_code_string.replace("```", '').replace("python", '')
print(clean_corrected_code)

In [ ]:
corrected_code_string = "```" + corrected_code_string.split("```")[1]
clean_corrected_code = corrected_code_string.replace("```python", "").replace("```", "").strip()

print("Passing corrected script back to the background COIN-OR CBC math solver...")
retry_results = execute_pyomo_code(clean_corrected_code)

if retry_results["success"]:
    print("\nSELF-CORRECTION SUCCESSFUL!")
    print("--- STDOUT (Solution Results) ---")
    print(retry_results["stdout"])
else:
    print("\nSELF-CORRECTION FAILED AGAIN!")
    print("--- STDERR (New Traceback Logs) ---")
    print(retry_results["stderr"])

# Self-correction works

Now we need to try to test the whole thing with the nl4opt_test dataset. 

In [ ]:
TEST_LIMIT = 10  # Evaluate the first 10 items to verify stability
MAX_CORRECTION_ATTEMPTS = 3 

successful_oneshot_count = 0
successful_corrected_count = 0

print(f"Launching evaluation over {TEST_LIMIT} problems...")

for idx, item in enumerate(test_problems[:TEST_LIMIT]):
    print(f"\nProcessing Problem {idx+1}/{TEST_LIMIT}...")
    problem_description = item.get("document", item.get("problem_text", ""))
    
    # --- PHASE 1: FIVE-ELEMENT FORMULATION ---
    f_prompt = generate_prompt.Q2F(problem_description)
    
    f_inputs = tokenizer([tokenizer.apply_chat_template([{"role": "user", "content": f_prompt}], tokenize=False, add_generation_prompt=True)], return_tensors="pt").to("cuda")
    with torch.no_grad():
        f_outputs = model.generate(**f_inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
    five_element_text = tokenizer.decode(f_outputs[0][len(f_inputs[0]):], skip_special_tokens=True).strip()
    
    # --- PHASE 2: INITIAL CODE GENERATION ---
    code_gen_prompt = generate_prompt.F2C(five_element_text)
    
    c_inputs = tokenizer([tokenizer.apply_chat_template([{"role": "user", "content": code_gen_prompt}], tokenize=False, add_generation_prompt=True)], return_tensors="pt").to("cuda")
    with torch.no_grad():
        c_outputs = model.generate(**c_inputs, max_new_tokens=1024, temperature=0.0, do_sample=False)
    current_code = tokenizer.decode(c_outputs[0][len(c_inputs[0]):], skip_special_tokens=True).strip()
    
    # --- PHASE 3: EXECUTION & AUTO-TESTING SELF CORRECTION LOOP ---
    attempt = 0
    solved_successfully = False
    oneshot_win = False
    
    while attempt < MAX_CORRECTION_ATTEMPTS:
        # Clean up any leftover markdown block syntax before execution
        clean_code = current_code.replace("```python", "").replace("```", "").strip()
        run_res = execute_pyomo_code(clean_code)
        
        if run_res["success"]:
            solved_successfully = True
            if attempt == 0:
                oneshot_win = True
            break
        else:
            attempt += 1
            # Run Self-Correction Generation Pass
            correction_prompt_input = {
                'ques': problem_description, 
                'five': five_element_text, 
                'code': clean_code, 
                'output': run_res["stdout"], 
                'error': run_res["stderr"]
            }
            corr_prompt = self_correction_prompt.self_correction(**correction_prompt_input)
            corr_inputs = tokenizer([tokenizer.apply_chat_template([{"role": "user", "content": corr_prompt}], tokenize=False, add_generation_prompt=True)], return_tensors="pt").to("cuda")
            
            with torch.no_grad():
                corr_outputs = model.generate(**corr_inputs, max_new_tokens=1024, temperature=0.0, do_sample=False)
            raw_response = tokenizer.decode(corr_outputs[0][len(corr_inputs[0]):], skip_special_tokens=True).strip()
            
            # Extract only the pure python code block from the critique template
            if "```python" in raw_response:
                current_code = raw_response.split("```python")[1].split("```")[0].strip()
            elif "```" in raw_response:
                current_code = raw_response.split("```")[1].split("```")[0].strip()
            else:
                current_code = raw_response.strip()

    if oneshot_win:
        successful_oneshot_count += 1
    if solved_successfully:
        successful_corrected_count += 1
    
    print(f"↳ Result - OneShot Win: {oneshot_win} | Final Success: {solved_successfully} (Attempts: {attempt})")

# --- RESULTS SUMMARY (Indentation Fixed) ---
print("\n==================== ACTUAL TASK 1 METRICS ====================")
print(f"Total Problems Processed: {TEST_LIMIT}")
print(f"Accuracy WITHOUT Self-Correction: {(successful_oneshot_count/TEST_LIMIT)*100:.2f}%")
print(f"Accuracy WITH Self-Correction: {(successful_corrected_count/TEST_LIMIT)*100:.2f}%")
print("================================================================")

# Report
reasons for the delta: Used quantization/compression techniques so even though running the test directly on ant-opt/LLMOPT-Qwen2.5-14B which is already fine tuned for this specific task, the model im running should have less precision than the real fine-tuned model.  
Sample Size Slice: I only test the model using a small portion of the dataset so the result might not be accurate

In [ ]:
import subprocess
# inference to get five elements
def infer_five_elem(question):
    messages = [
        {"role": "user", "content": generate_prompt.Q2F(question)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None


# inference to get pyomo python code
def infer_code(five_elem):
    messages = [
        {"role": "user", "content": generate_prompt.F2C(five_elem)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')


# execute the code
def test_code(code_str):
    code_path = f"./test.py"
    with open(code_path, "w") as f1:
        f1.write(code_str)

    ans = subprocess.run(f"python {code_path}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # return answer logs, error code
    return str(ans.stdout.decode('gbk', errors='ignore')), str(ans.stderr.decode('gbk', errors='ignore'))

In [ ]:
# this one is not from the official repo
def self_correct(ques, five, code, output, error):
    messages = [
        {"role": "user", "content": self_correction_prompt.self_correction(ques, five, code, output, error)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans


def self_correct_analysis_five_elem_prompt(analysis, ques, five):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The problem is as follows.

{ques}

The five-element formulation is as follows.

{five}

Based on the analysis. Please write the corresponding five-element model. Please use LaTeX and ``` plain text environment to complete the following template to model the above optimization problem into five elements: 

```
## Sets: 
[You need to fill in]

## Parameters: 
[You need to fill in]

## Variables: 
[You need to fill in]

## Objective: 
[You need to fill in]

## Constraints: 
[You need to fill in]
```
"""

    
def self_correct_analysis_code_prompt(analysis, five, code):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The five-element formulation is as follows.

{five}

The code is as follows.

{code}

Based on the analysis. Please write the corresponding Pyomo code. Please add `from pyomo.environ import *` at the beginning of your code (You can add other `import` as well). Please print the optimal solution and the value of the objective function. Please do not output the running log. You need to write it in the form of a class and add a main function: 

```python
[write your code here]
```
"""

def self_correct_analysis_five_elem(analysis, ques, five):
    messages = [
        {"role": "user", "content": self_correct_analysis_five_elem_prompt(analysis, ques, five)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None

def self_correct_analysis_code(analysis, five, code):
    messages = [
        {"role": "user", "content": self_correct_analysis_code_prompt(analysis, five, code)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')

In [ ]:
five_elem = infer_five_elem(problem_description)

In [ ]:
print(five_elem)

In [ ]:
inf_code = infer_code(five_elem)

In [ ]:
print(inf_code)

In [ ]:
t_code_success = test_code(inf_code)
t_code_fail = test_code(inf_code[0:10])

In [ ]:
for i, item in enumerate(t_code_success):
    print(i, item)
for i, item in enumerate(t_code_fail):
    print(i, item)

print(item.split('\n\n\n'))
print(sample_item['answer'])


In [ ]:
code = self_correct(problem_description, five_elem, inf_code[0:10], t_code_fail[0], t_code_fail[1])

In [ ]:
print(code)

In [ ]:
print(self_correct_analysis_prompt(problem_description, five_elem, inf_code[0:10], code))


In [ ]:
real_code = self_correct_analysis(problem_description, five_elem, inf_code[0:10], code)

In [ ]:
print(real_code)

In [ ]:
TEST_LIMIT = 10  # Evaluate 10 test problems
MAX_CORRECTION_ATTEMPTS = 3 
START_INDEX = 20
END_INDEX = TEST_LIMIT + START_INDEX

print(f"Launching real-time evaluation over {TEST_LIMIT} problems...")

real_time_data = test_problems[START_INDEX:END_INDEX]

successful_oneshot_count = 0
successful_corrected_count = 0
for i, item in enumerate(real_time_data):
    print(f"\nProcessing Problem {i+1}/{TEST_LIMIT}...")
    problem_description = item['question']
    problem_answer = str(item['answer'])
    print('question = ' + problem_description)
    print("answer = " + problem_answer)
    # --- PHASE 1: FIVE-ELEMENT FORMULATION ---  
    five_element_text = infer_five_elem(problem_description)
    
    # --- PHASE 2: INITIAL CODE GENERATION ---
    code = infer_code(five_element_text)
   
    # --- PHASE 3: EXECUTION & AUTO-TESTING SELF CORRECTION LOOP ---
    attempt = 0
    solved_successfully = False
    oneshot_win = False
    wrong_code = []

    while attempt < MAX_CORRECTION_ATTEMPTS:
        # Clean up any leftover markdown block syntax before execution
        run_res = test_code(code)

        print(f"attempt {attempt} output: {run_res[0]}")
        check_answer = problem_answer in run_res[0]

        if not run_res[1] and check_answer:
            solved_successfully = True
            if attempt == 0:
                oneshot_win = True
            break
        else:
            attempt += 1
            wrong_code.append(code)
            # Run Self-Correction Generation Pass
            correction_prompt_input = {
                'ques': problem_description, 
                'five': five_element_text, 
                'code': code, 
                'output': run_res[0], 
                'error': run_res[1]
            }
            
            self_correct_output = self_correct(**correction_prompt_input)
            five_element_text = self_correct_analysis_five_elem(self_correct_output, problem_description, five_element_text)
            code = self_correct_analysis_code(self_correct_output, five_element_text, code)

    if oneshot_win:
        successful_oneshot_count += 1
    if solved_successfully:
        successful_corrected_count += 1
    
    print(f"↳ Result - OneShot Win: {oneshot_win} | Final Success: {solved_successfully} (Attempts: {attempt})")
    if wrong_code:
        for i, wc in enumerate(wrong_code):
            print("================================================================")
            print(f"Attempt {i}:\n\n{wc}")
            print("================================================================")

# --- RESULTS SUMMARY (Indentation Fixed) ---
print("\n==================== ACTUAL TASK 1 METRICS ====================")
print(f"Total Problems Processed: {TEST_LIMIT}")
print(f"Accuracy WITHOUT Self-Correction: {(successful_oneshot_count/TEST_LIMIT)*100:.2f}%")
print(f"Accuracy WITH Self-Correction: {(successful_corrected_count/TEST_LIMIT)*100:.2f}%")
print("================================================================")

# Task 2

In [ ]:
dataset_path = "LLMOPT/data/testset/industryor.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems_or = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems_or)} test items from industryor.")
print(f"Sample test problem:\n{json.dumps(test_problems_or[0], indent=1)}")

In [ ]:
or_data = test_problems_or[START_INDEX:END_INDEX]
or_data

In [ ]:
def modified_Q2F(ques):
    ques = generate_prompt.bound_symbol + ques + generate_prompt.bound_symbol

    dialogue_instruction = """You are acting as an interactive Operations Research Consultant. The user has provided an initial problem statement, followed by a running conversation history. 
Your job is to progress towards building a mathematically sound five-element optimization model by eliciting missing details or checking for logical contradictions.

CRITICAL LOGICAL RULES:
1. GROUNDING RULE: If information, sets, parameters, or numeric values are not explicitly stated by the user, you MUST mark them as [MISSING]. Do NOT invent, guess, or hallucinate arbitrary variables, costs, constraints, or numbers.
2. QUESTION ISOLATION RULE: When asking a clarifying question, base it strictly on what is missing. Do not invent anything if the user has not explicitly mentioned them. And Do not ask anything the user has explicitly mentioned.
"""
    
    modified_five_suffix = """
STRICT STATUS DEFINITIONS:
- If any elements or numerical parameters required to build the model are unknown or marked MISSING, you MUST set Status under Next Action to INCOMPLETE.
- If and only if ALL five elements are fully collected, clear, and have zero MISSING tags, you MUST set Status under Next Action to COMPLETE. 
- If you detect that the user's requirements or stated constraints are mathematically impossible or directly contradict each other, set Status to CONFLICT.

STRICT ANSWER FORMAT:
- Please analyze the problem description and write the corresponding five-element model. 
- Please use json and {} in string format to complete the following template to model the above optimization problem into five elements: 

{
    "sets": "If elements are missing, write MISSING only. Otherwise you need to fill in ["string"] format",
    "parameters": "If elements are missing, write MISSING only. Otherwise you need to fill in ["string"] format",
    "variables": "If elements are missing, write MISSING only. Otherwise you need to fill in ["string"] format",
    "objective": "If elements are missing, write MISSING only. Otherwise you need to fill in "string",
    "constraints": "If elements are missing, write MISSING only. Otherwise you need to fill in ["string"] format",
    "status": "INCOMPLETE / COMPLETE / CONFLICT"
    "conflicts": "None / If True, describe the mutual infeasibility clearly"
    "question": "If Status is INCOMPLETE, ask exactly ONE clarifying question. If Status is COMPLETE, write None"
}
"""

    return generate_prompt.five_description_complex + generate_prompt.ques_description_five + ques + dialogue_instruction + modified_five_suffix

# TASK 2 Build a Multi-Turn Conversational Layer on Top of LLMOPT

In [ ]:
!nvidia-smi

Tue Jun  2 07:08:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cloning the Repository

In [ ]:
!git clone https://github.com/caigaojiang/LLMOPT

Cloning into 'LLMOPT'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 141 (delta 58), reused 46 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.47 MiB | 5.32 MiB/s, done.
Resolving deltas: 100% (58/58), done.


## Installing Dependencies

In [ ]:
!pip install transformers==4.42.0 \
             accelerate==0.27.2 \
             tiktoken==0.7.0 \
             einops==0.7.0 \
             transformers_stream_generator==0.0.4 \
             peft==0.11.1 \
             trl==0.9.6 \
             Jinja2==3.1.4 \
             bitsandbytes \
             "numpy<=2.0.0 "\
             pyomo
# Install the Linux system-level math solvers that Pyomo relies on
!apt-get install -y coinor-cbc glpk-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


## Load Necessary modules

In [ ]:
import os
import json
import sys


import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)

import peft
import trl
from pyomo.environ import *
import pyomo.environ as aml

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Current GPU: Tesla T4


In [ ]:
# Load the necessary prompt modules from the official LLOMPT repo to build prompts
module_path = os.path.abspath('LLMOPT/prompts')
print(f"module_path: {module_path}")
# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

import generate_prompt
import self_correction_prompt

module_path: /content/LLMOPT/prompts


## Load the Model
Use quantization to compress the model

In [ ]:
# Configure the 4-bit quantization layout to protect cloud VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "ant-opt/LLMOPT-Qwen2.5-14B"

# Download and cache the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Download, quantize, and load the 14B model safely
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True        # Forces shard-by-shard loading directly into RAM/VRAM
)

print("done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

model-00011-of-00012.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00012-of-00012.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

done


## Load the industryor Dataset

In [ ]:
dataset_path = "LLMOPT/data/testset/inustryor.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems)} test items from industryor.")
print(f"Sample test problem:\n{json.dumps(test_problems[0], indent=1)}")

Successfully loaded 230 test items from NL4Opt.
Sample test problem:
{
 "question": "There has been an oil spill in the ocean and ducks need to be taken to shore to be cleaned either by boat or by canoe. A boat can take 10 ducks per trip while a canoe can take 8 ducks per trip. Since the boats are motor powered, they take 20 minutes per trip while the canoes take 40 minutes per trip. In order to avoid further environmental damage, there can be at most 12 boat trips and at least 60% of the trips should be by canoe. If at least 300 ducks need to be taken to shore, how many of each transportation method should be used to minimize the total amount of time needed to transport the ducks?",
 "answer": 1160,
 "ori": "5_nl4opt_test",
 "index": 1
}


## Testing and Evaluating the Model

### Copy the Necessary Functions from the official LLOMPT REPO And Create New functions to do the Self Correction Loop

In [ ]:
import subprocess
# inference to get five elements
def infer_five_elem(question):
    messages = [
        {"role": "user", "content": generate_prompt.Q2F(question)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None


# inference to get pyomo python code
def infer_code(five_elem):
    messages = [
        {"role": "user", "content": generate_prompt.F2C(five_elem)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')


# execute the code
def test_code(code_str):
    code_path = f"./test.py"
    with open(code_path, "w") as f1:
        f1.write(code_str)

    ans = subprocess.run(f"python {code_path}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # return answer logs, error code
    return str(ans.stdout.decode('gbk', errors='ignore')), str(ans.stderr.decode('gbk', errors='ignore'))

In [ ]:
# this one is not from the official repo
def self_correct(ques, five, code, output, error):
    messages = [
        {"role": "user", "content": self_correction_prompt.self_correction(ques, five, code, output, error)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans


def self_correct_analysis_five_elem_prompt(analysis, ques, five):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The problem is as follows.

{ques}

The five-element formulation is as follows.

{five}

Based on the analysis. Please write the corresponding five-element model. Please use LaTeX and ``` plain text environment to complete the following template to model the above optimization problem into five elements: 

```
## Sets: 
[You need to fill in]

## Parameters: 
[You need to fill in]

## Variables: 
[You need to fill in]

## Objective: 
[You need to fill in]

## Constraints: 
[You need to fill in]
```
"""

    
def self_correct_analysis_code_prompt(analysis, five, code):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The five-element formulation is as follows.

{five}

The code is as follows.

{code}

Based on the analysis. Please write the corresponding Pyomo code. Please add `from pyomo.environ import *` at the beginning of your code (You can add other `import` as well). Please print the optimal solution and the value of the objective function. Please do not output the running log. You need to write it in the form of a class and add a main function: 

```python
[write your code here]
```
"""

def self_correct_analysis_five_elem(analysis, ques, five):
    messages = [
        {"role": "user", "content": self_correct_analysis_five_elem_prompt(analysis, ques, five)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None

def self_correct_analysis_code(analysis, five, code):
    messages = [
        {"role": "user", "content": self_correct_analysis_code_prompt(analysis, five, code)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')

### Running the model

In [ ]:
TEST_LIMIT = 20  # Evaluate 10 test problems
MAX_CORRECTION_ATTEMPTS = 3 
START_INDEX = 20
END_INDEX = TEST_LIMIT + START_INDEX

print(f"Launching real-time evaluation over {TEST_LIMIT} problems...")

real_time_data = test_problems[START_INDEX:END_INDEX]

successful_oneshot_count = 0
successful_corrected_count = 0
for i, item in enumerate(real_time_data):
    print(f"\nProcessing Problem {i+1}/{TEST_LIMIT}...")
    problem_description = item['question']
    problem_answer = str(item['answer'])
    print('question = ' + problem_description)
    print("answer = " + problem_answer)
    # --- PHASE 1: FIVE-ELEMENT FORMULATION ---  
    five_element_text = infer_five_elem(problem_description)
    
    # --- PHASE 2: INITIAL CODE GENERATION ---
    code = infer_code(five_element_text)
   
    # --- PHASE 3: EXECUTION & AUTO-TESTING SELF CORRECTION LOOP ---
    attempt = 0
    solved_successfully = False
    oneshot_win = False
    wrong_code = []

    while attempt < MAX_CORRECTION_ATTEMPTS:
        # Clean up any leftover markdown block syntax before execution
        run_res = test_code(code)

        print(f"attempt {attempt} output: {run_res[0]}")
        check_answer = problem_answer in run_res[0]

        if not run_res[1] and check_answer:
            solved_successfully = True
            if attempt == 0:
                oneshot_win = True
            break
        else:
            attempt += 1
            wrong_code.append(code)
            # Run Self-Correction Generation Pass
            correction_prompt_input = {
                'ques': problem_description, 
                'five': five_element_text, 
                'code': code, 
                'output': run_res[0], 
                'error': run_res[1]
            }
            
            self_correct_output = self_correct(**correction_prompt_input)
            five_element_text = self_correct_analysis_five_elem(self_correct_output, problem_description, five_element_text)
            code = self_correct_analysis_code(self_correct_output, five_element_text, code)

    if oneshot_win:
        successful_oneshot_count += 1
    if solved_successfully:
        successful_corrected_count += 1
    
    print(f"↳ Result - OneShot Win: {oneshot_win} | Final Success: {solved_successfully} (Attempts: {attempt})")
    if wrong_code:
        for i, wc in enumerate(wrong_code):
            print("================================================================")
            print(f"Attempt {i}:\n\n{wc}")
            print("================================================================")

# --- RESULTS SUMMARY (Indentation Fixed) ---
print("\n==================== ACTUAL TASK 1 METRICS ====================")
print(f"Total Problems Processed: {TEST_LIMIT}")
print(f"Accuracy WITHOUT Self-Correction: {(successful_oneshot_count/TEST_LIMIT)*100:.2f}%")
print(f"Accuracy WITH Self-Correction: {(successful_corrected_count/TEST_LIMIT)*100:.2f}%")
print("================================================================")

Launching real-time evaluation over 10 problems...

Processing Problem 1/10...
question = A bakery bakes bagels and croissants. A batch of bagels can be made using 2 hours of oven time and 0.25 hours of pastry chef time. A batch of croissants is more complicated, so while they take 1 hour of oven time, they take 2 hours of pastry chef time. In a day, the bakery has at most 70 hours available for the oven and 32 pastry chef hours available. Using all the available capacity, what is the maximum profit the bakery can generate assuming the profit per batch is $20 and $40 respectively for a batch of bagels and a batch of croissants.
answer = 1060
attempt 0 output: Status: ok
Termination Condition: optimal
Optimal Solution Found
Objective Value (Total Profit): 1060.0
Batch Production Levels:
  Bagels: 29.0
  Croissants: 12.0

↳ Result - OneShot Win: True | Final Success: True (Attempts: 0)

Processing Problem 2/10...
question = Platinum in combination with palladium has been used as a cataly

## Report
Reasons for the delta: Used quantization/compression techniques so even though running the test directly on ant-opt/LLMOPT-Qwen2.5-14B which is already fine tuned for this specific task, the model im running should have less precision than the real fine-tuned model.  
And since I only test the model using a small portion of the dataset so the result is more likely to not be accurate. My self correction loop might not be as good as the ones the author made and the I set the max attempt to 3 instead of 12. 

In [ ]:

!git clone https://github.com/caigaojiang/LLMOPT

Cloning into 'LLMOPT'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (137/137), done.objects:  65% (90/137)
remote: Total 141 (delta 58), reused 46 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.47 MiB | 11.24 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [ ]:
!pip install transformers==4.42.0 \
             accelerate==0.27.2 \
             tiktoken==0.7.0 \
             einops==0.7.0 \
             transformers_stream_generator==0.0.4 \
             peft==0.11.1 \
             trl==0.9.6 \
             Jinja2==3.1.4 \
             bitsandbytes \
            
             pyomo
# Install the Linux system-level math solvers that Pyomo relies on
!apt-get install -y coinor-cbc glpk-utils

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-stream-generator-0.0.4.tar.gz (12 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached peft-0.11.1-py3-none-any.whl.metadata (13 kB)
  Using cached trl-0.9.6-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 3.9 MB/s  0:00:04 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached datasets-4.8.5-py3-none

In [ ]:
import os
import json
import sys

# import torch
# from transformers import (
#     AutoModelForCausalLM, 
#     AutoTokenizer, 
#     BitsAndBytesConfig,
#     pipeline
# )

# import peft
# import trl
# from pyomo.environ import *
# import pyomo.environ as aml

# print(f"CUDA Available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"Current GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configure the 4-bit quantization layout to protect cloud VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "ant-opt/LLMOPT-Qwen2.5-14B"

# Download and cache the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Download, quantize, and load the 14B model into active GPU memory
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("done")

In [ ]:
# Build an absolute path from this notebook's parent directory
module_path = os.path.abspath('LLMOPT/prompts')
print(f"module_path: {module_path}")
# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

import generate_prompt
import self_correction_prompt

module_path: /Users/marcoeditiahusin/alpha-z/LLMOPT/prompts


In [ ]:
dataset_path = "LLMOPT/data/testset/industryor.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems)} test items from industryor.")
print(f"Sample test problem:\n{json.dumps(test_problems[0], indent=1)}")

Successfully loaded 100 test items from industryor.
Sample test problem:
{
 "question": "The Zhang family has 6 children, Harry, Hermione, Ron, Fred, George, and Ginny. The cost of taking Harry is $1200, Hermione is $1650, Ron is $750, Fred is $800, George is $800, and Ginny is $1500. Which children should the couple take to minimize the total cost of taking the children?\n\nThey can take a maximum of 4 children on the upcoming trip.\n\nGinny is the youngest, so the Zhang family will definitely take her.\n\nIf the couple takes Harry, they will not take Fred because Harry doesn't get along with him.\n\nIf the couple takes Harry, they will not take George because Harry doesn't get along with him.\n\nIf they take George, they must also take Fred.\n\nIf they take George, they must also take Hermione.\n\nAlthough this will cost them a lot of money, the Zhang family has decided to take at least three children.",
 "answer": 3050,
 "ori": "6_industryor",
 "index": 1
}


In [ ]:
print(modified_Q2F(test_problems_or[0]['question']))

In [ ]:
def modified_infer_five_elem(question):
    messages = [
        {"role": "user", "content": modified_Q2F(question)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    print(f"Raw: {response}")
    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return response

In [ ]:
x = modified_infer_five_elem(test_problems_or[0]['question'])

In [ ]:
print(x)

In [ ]:
y = modified_infer_five_elem('help me schedule my delivery drivers this week"')

In [ ]:
print(y)

In [ ]:
z = modified_infer_five_elem('hey dude how you doing help me schedule my task my first task is to eat salmon second task to eat banana third task is to eat salmon can you please schedule it so i can achieve all of em in one day. task 1 takes 1 min and task 3 takes 5 minutes while task 2 takes 1 minute')

In [ ]:
print(z)

In [ ]:
import json

def build_message(role, content):
    return {"role": role, "content": content}

def conversation(vague_problem_statement):
    # Initialize the chat history with system rules and the initial vague prompt
    chat_history = [
        build_message("system", "You are an assistant acting as an interactive Operations Research Consultant. The user has provided an initial problem statement, followed by a running conversation history. Your job is to progress towards building a mathematically sound five-element optimization model by eliciting missing details or checking for logical contradictions."),
        build_message("user", vague_problem_statement)
    ]
    
    print(f"\n[Agent]: Analyzing your initial problem...")
    response_json = modified_infer_five_elem(chat_history)
    chat_history.append(build_message("assistant"), response_json)
    while True:
        agent_data = json.loads(response_json)
        
        # Check if constraint conflict detection flagged an issue
        if agent_data.get("conflicts") or agent_data.get("conflicts") != "None":
            print(f"\n⚠️ [Conflict Detected]: {agent_data['conflicts']}")
            
        # Check for the Exit Condition
        if agent_data.get("question") == "None" or status = "COMPLETE" or not agent_data.get("question"):
            print("\n✅ [Agent]: Formulation complete! Extracting Five-Element Model...")
            return response_json
            
        # 4. Otherwise, continue the conversation
        agent_question = agent_data["agentResponse"]
        print(f"\n[Agent]: {agent_question}")
        chat_history.append(build_message("assistant", agent_question))
        
        # 5. Get the user's clarifying answer
        user_answer = input("Your answer (or type 'quit' to exit): ")
        if user_answer.strip().lower() == 'quit':
            print("exited")
            return None
        else: 
            
            
        chat_history.append(build_message("user", user_answer))